In [1]:
!nvidia-smi

Wed May  6 23:52:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile matrix_mul.cu

#include <iostream>
using namespace std;

#define N 3

__global__ void matrixMul(int A[N][N], int B[N][N], int C[N][N])
{
    int row = threadIdx.y;
    int col = threadIdx.x;

    int sum = 0;

    for(int k = 0; k < N; k++)
    {
        sum += A[row][k] * B[k][col];
    }

    C[row][col] = sum;
}

int main()
{
    int A[N][N] = {
        {1,2,3},
        {4,5,6},
        {7,8,9}
    };

    int B[N][N] = {
        {1,0,0},
        {0,1,0},
        {0,0,1}
    };

    int C[N][N];

    int (*d_A)[N], (*d_B)[N], (*d_C)[N];

    cudaMalloc((void**)&d_A, sizeof(int)*N*N);
    cudaMalloc((void**)&d_B, sizeof(int)*N*N);
    cudaMalloc((void**)&d_C, sizeof(int)*N*N);

    cudaMemcpy(d_A, A, sizeof(int)*N*N, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, sizeof(int)*N*N, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(N, N);

    matrixMul<<<1, threadsPerBlock>>>(d_A, d_B, d_C);

    cudaMemcpy(C, d_C, sizeof(int)*N*N, cudaMemcpyDeviceToHost);

    cout << "Matrix Multiplication Result:\n";

    for(int i=0; i<N; i++)
    {
        for(int j=0; j<N; j++)
        {
            cout << C[i][j] << " ";
        }
        cout << endl;
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing matrix_mul.cu


In [3]:
!nvcc matrix_mul.cu -o matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./matrix_mul

Matrix Multiplication Result:
1 2 3 
4 5 6 
7 8 9 
